# Speech Psychology — LightGBM روی BERT Embeddings (Google Colab)
**روش:** Sentence Embeddings (all-MiniLM-L6-v2) + LightGBM

بهبود نسبت به مرحله قبل: به جای Ridge Regression از LightGBM استفاده می‌کنیم
که می‌تواند روابط غیرخطی بین embedding ها و لیبل‌ها را یاد بگیرد.

**قبل از شروع:** از منوی بالا `Runtime > Change runtime type > T4 GPU` رو انتخاب کن.

## 1. Install & Import

In [ ]:
!pip install sentence-transformers lightgbm scikit-learn pandas numpy matplotlib -q
print('Done!')

In [ ]:
# بررسی GPU
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('⚠️ GPU فعال نیست — برو Runtime > Change runtime type > T4 GPU')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time, os

from sentence_transformers import SentenceTransformer
import lightgbm as lgb
from sklearn.model_selection import KFold
from sklearn.metrics import root_mean_squared_error

import warnings
warnings.filterwarnings('ignore')
print('Libraries loaded!')

## 2. Mount Drive & Upload Data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/speech_psychology'
os.makedirs(PROJECT_DIR, exist_ok=True)
print(f'Project dir: {PROJECT_DIR}')

In [ ]:
# اگه train.csv و test.csv قبلاً توی Drive هستن این سلول رو skip کن
from google.colab import files
import shutil

print('فایل‌های train.csv و test.csv رو انتخاب کن:')
uploaded = files.upload()
for fname in uploaded:
    shutil.copy(fname, f'{PROJECT_DIR}/{fname}')
    print(f'✅ {fname} copied to Drive')

## 3. Load Data

In [ ]:
train = pd.read_csv(f'{PROJECT_DIR}/train.csv')
test  = pd.read_csv(f'{PROJECT_DIR}/test.csv')

TARGET_COLS = [
    'sense', 'honor', 'curse', 'despise', 'situation',
    'antihuman', 'roughness', 'slaughter', 'strike_support', 'depression_rate'
]

X_train_text = train['text'].fillna('').astype(str).tolist()
y_train      = train[TARGET_COLS].values
X_test_text  = test['text'].fillna('').astype(str).tolist()

print(f'Train: {len(X_train_text)} samples')
print(f'Test:  {len(X_test_text)} samples')

## 4. Generate Embeddings

اگه از notebook مرحله قبل embedding ها رو توی Drive ذخیره کردی،
این سلول خودکار اون‌ها رو لود می‌کنه و نیازی به encode مجدد نیست.

In [ ]:
TRAIN_EMB_PATH = f'{PROJECT_DIR}/train_embeddings.npy'
TEST_EMB_PATH  = f'{PROJECT_DIR}/test_embeddings.npy'

if os.path.exists(TRAIN_EMB_PATH) and os.path.exists(TEST_EMB_PATH):
    print('Loading saved embeddings from Drive...')
    X_train_emb = np.load(TRAIN_EMB_PATH)
    X_test_emb  = np.load(TEST_EMB_PATH)
    print('Loaded!')
else:
    print('Loading embedding model...')
    embedder = SentenceTransformer('all-MiniLM-L6-v2', device=device)

    print('Encoding train set...')
    t0 = time.time()
    X_train_emb = embedder.encode(
        X_train_text, batch_size=256,
        show_progress_bar=True, convert_to_numpy=True, device=device
    )
    print(f'Train encoded in {(time.time()-t0)/60:.1f} min')

    print('Encoding test set...')
    t0 = time.time()
    X_test_emb = embedder.encode(
        X_test_text, batch_size=256,
        show_progress_bar=True, convert_to_numpy=True, device=device
    )
    print(f'Test encoded in {(time.time()-t0)/60:.1f} min')

    np.save(TRAIN_EMB_PATH, X_train_emb)
    np.save(TEST_EMB_PATH,  X_test_emb)
    print('Embeddings saved to Drive!')

print(f'Train embeddings: {X_train_emb.shape}')
print(f'Test embeddings:  {X_test_emb.shape}')

## 5. Train LightGBM — یه مدل به ازای هر ستون

برخلاف Ridge که یه مدل linear بود، LightGBM یه **Gradient Boosting** مدله:
- درخت‌های تصمیم رو پشت سر هم می‌سازه
- هر درخت خطای درخت قبلی رو کم می‌کنه
- روابط غیرخطی رو یاد می‌گیره

برای هر کدام از ۱۰ ستون هدف، یه مدل جداگانه آموزش می‌دهیم.

In [ ]:
# پارامترهای LightGBM
lgbm_params = {
    'objective':      'regression',   # مسئله regression داریم
    'metric':         'rmse',
    'learning_rate':  0.05,           # گام یادگیری
    'num_leaves':     63,             # پیچیدگی هر درخت
    'min_child_samples': 20,          # جلوگیری از overfit
    'subsample':      0.8,            # هر درخت روی 80% داده آموزش می‌بیند
    'colsample_bytree': 0.8,          # هر درخت 80% feature ها رو می‌بیند
    'n_estimators':   500,
    'verbose':        -1
}

kf = KFold(n_splits=5, shuffle=True, random_state=42)
models   = {}   # مدل نهایی هر ستون
rmse_scores = {}

for col_idx, col in enumerate(TARGET_COLS):
    y_col = y_train[:, col_idx]
    fold_rmses = []

    for fold, (train_idx, val_idx) in enumerate(kf.split(X_train_emb)):
        X_tr, X_val = X_train_emb[train_idx], X_train_emb[val_idx]
        y_tr, y_val = y_col[train_idx],        y_col[val_idx]

        model = lgb.LGBMRegressor(**lgbm_params)
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            callbacks=[lgb.early_stopping(50, verbose=False),
                       lgb.log_evaluation(period=-1)]
        )
        preds_val = model.predict(X_val)
        rmse = root_mean_squared_error(y_val, preds_val)
        fold_rmses.append(rmse)

    # آموزش مدل نهایی روی همه داده train
    final_model = lgb.LGBMRegressor(**lgbm_params)
    final_model.fit(X_train_emb, y_col)
    models[col] = final_model

    rmse_scores[col] = np.mean(fold_rmses)
    print(f'{col:20s}: RMSE = {rmse_scores[col]:.4f}')

mcrmse = np.mean(list(rmse_scores.values()))
score  = (1.5 - mcrmse) * (100/150) * 150
print(f'\nMCRMSE: {mcrmse:.4f}')
print(f'Estimated Score: {score:.2f} / 150')

## 6. مقایسه با مدل‌های قبلی

In [ ]:
baseline_mcrmse = 0.7757   # TF-IDF + Ridge
bert_ridge_mcrmse = None   # عدد رو از notebook مرحله قبل اینجا بذار

results_df = pd.DataFrame({
    'Column': list(rmse_scores.keys()),
    'RMSE':   list(rmse_scores.values())
}).sort_values('RMSE', ascending=False)

fig, ax = plt.subplots(figsize=(10, 4))
ax.barh(results_df['Column'], results_df['RMSE'], color='steelblue', edgecolor='white')
ax.axvline(x=1.5,               color='red',    linestyle='--', label='Reject threshold (1.5)')
ax.axvline(x=baseline_mcrmse,   color='orange', linestyle='--', label=f'Baseline MCRMSE ({baseline_mcrmse})')
ax.axvline(x=mcrmse,            color='green',  linestyle='--', label=f'LightGBM MCRMSE ({mcrmse:.3f})')
ax.set_xlabel('RMSE')
ax.set_title('RMSE per Column — LightGBM + BERT Embeddings')
ax.legend()
plt.tight_layout()
plt.show()

print('\n--- مقایسه مدل‌ها ---')
print(f'Baseline (TF-IDF + Ridge):  MCRMSE = {baseline_mcrmse:.4f}')
print(f'LightGBM + BERT Embeddings: MCRMSE = {mcrmse:.4f}')
print(f'بهبود: {(baseline_mcrmse - mcrmse):.4f}')

## 7. Predict & Save Output

In [ ]:
# پیش‌بینی هر ستون با مدل مربوطه
preds = np.column_stack([
    models[col].predict(X_test_emb) for col in TARGET_COLS
])

# clip به بازه [0, 4]
preds = np.clip(preds, 0, 4)

output = pd.DataFrame(preds, columns=TARGET_COLS).round(3)
print(f'Output shape: {output.shape}')  # باید (1754, 10) باشد
output.head()

In [ ]:
OUTPUT_PATH = f'{PROJECT_DIR}/output_lgbm.csv'
output.to_csv(OUTPUT_PATH, index=False)
print(f'output_lgbm.csv saved — {len(output)} rows')

# دانلود مستقیم
from google.colab import files
files.download(OUTPUT_PATH)

## 8. خلاصه

| مدل | روش | MCRMSE |
|-----|-----|--------|
| Baseline | TF-IDF + Ridge | 0.776 |
| مرحله ۲ | BERT Embedding + Ridge | ؟ |
| این مدل | BERT Embedding + LightGBM | ؟ |

**چرا LightGBM از Ridge بهتره؟**
- Ridge فقط روابط خطی یاد می‌گیره
- LightGBM می‌تونه روابط پیچیده‌تر و غیرخطی رو capture کنه
- با early stopping از overfit جلوگیری می‌کنه

**مرحله بعد (اختیاری):** Ensemble کردن چند مدل برای نتیجه بهتر